# Module 8: Automated Data Validation & Profiling in Python and R

**Unit D · Week 8** · Track 1 — Data Integration, Standards, Metadata & Quality

Codifies Module 7's quality dimensions as re-runnable rules with pandera, so every pipeline refresh is checked the same way automatically, and failing rows are flagged for review rather than silently dropped.

## Learning objectives

- Write automated, re-runnable validation rules: schema, ranges, and referential integrity against a P-code gazetteer.
- Generate an automated data-profiling first pass, and handle failures by flagging rows for review, never silently dropping them.


## Lesson content

- **Why codify validation.** Manual eyeballing doesn't scale once a pipeline refreshes monthly or more often — encoding the Module 7 quality checks as rules means every refresh is checked the same way, automatically.
- **pandera** for schema, range, and uniqueness checks defined declaratively against a dataframe (the R equivalent is `pointblank`, with a human-readable pass/fail report; shown in the R notebook for reference, not executed inline here since this repo doesn't install the full R stack — see the root README).
- **Handling failures responsibly.** Failing rows should be flagged and routed for review, never silently dropped — dropping bad rows quietly turns a data-quality problem into a data-completeness problem nobody knows about.

In [1]:
import pandera.pandas as pa
from pandera.pandas import Column, Check
import pandas as pd

gazetteer_codes = set(pd.read_csv("../../../data/raw/cod_ab_gazetteer.csv")["district_pcode"])

schema = pa.DataFrameSchema({
    "district_pcode": Column(str, Check.isin(gazetteer_codes), nullable=False),
    "value": Column(float, Check.ge(0)),
    "date": Column(str, Check.str_matches(r"^\d{4}-\d{2}-\d{2}$")),
})

df = pd.read_csv("../../../data/processed/harmonized_regional_data.csv", skiprows=[1])  # skip the HXL row
try:
    schema.validate(df, lazy=True)
    print("All validation checks passed.")
except pa.errors.SchemaErrors as err:
    print(err.failure_cases)   # flagged for review — not silently dropped
    err.failure_cases.to_csv("../../../data/processed/validation_failures.csv", index=False)

All validation checks passed.


## Your turn

Write a validation schema for the Module 4 harmonized dataset covering at least: no missing/unmatched P-codes, value >= 0, and ISO-format dates. Run it, review the flagged rows (there should be none, since Module 4 already corrected them), and confirm the pass.

**Formative assessment.** Submitted validation script plus before/after row counts, graded on rule coverage and correct handling of failures (flagged and resolved, not silently dropped).